# Personal Travel Planner Agent

Welcome to your AI-powered travel assistant! This notebook implements a multi-agent system that:
1. **Inquires** about your travel preferences.
2. **Researches** destinations, flights, and accommodations using the web.
3. **Synthesizes** a detailed, day-by-day itinerary.
4. **Delivers** the plan directly to your inbox.

In [ ]:
import os
import asyncio
import json
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

from agents import Agent, Runner, trace, function_tool, WebSearchTool
from agents.model_settings import ModelSettings


load_dotenv(override=True)

##  Models & Schemas
We use **Structured Outputs** (Pydantic) to ensure our agents return data in a predictable format.

In [ ]:
class ItineraryDay(BaseModel):
    day_number: int
    theme: str = Field(description="The main theme or focus of the day (e.g., 'Historical Treasures')")
    activities: List[str] = Field(description="A list of 3-4 specific activities or places to visit")
    estimated_cost: str = Field(description="Estimated budget for this day's activities")
    dining_suggestions: List[str] = Field(description="Recommendations for lunch and dinner")

class TravelItinerary(BaseModel):
    destination: str
    duration_days: int
    total_estimated_budget: str
    highlights: List[str] = Field(description="Top 3-5 highlights of the trip")
    daily_plans: List[ItineraryDay]
    travel_tips: List[str] = Field(description="Essential local tips (weather, customs, transport)")

class UserEngagement(BaseModel):
    is_planning_complete: bool = Field(description="Whether we have enough info to start research")
    destination: Optional[str]
    dates: Optional[str]
    interests: List[str]
    budget_level: Optional[str] = Field(description="Budget, Mid-range, or Luxury")

## Specialized Agents

In [ ]:
MODEL = "gpt-4o-mini"

inquiry_instructions = """
You are a friendly travel consultant. Your goal is to gather: 
1. Destination (City and Country)
2. Dates/Duration
3. Interests (e.g., Art, Hiking, Foodie, Relaxation)
4. Budget Level (Budget, Mid-range, Luxury)

If any of these are missing, ask for them. If all are present, set is_planning_complete to True.
"""
inquiry_agent = Agent(
    name="Inquiry Agent",
    instructions=inquiry_instructions,
    model=MODEL,
    output_type=UserEngagement
)

research_instructions = """
You are a destination researcher. Given a location and interests, find top attractions, 
hidden gems, and local logistics. Provide a concise summary of your findings for a report writer.
"""
research_agent = Agent(
    name="Research Agent",
    instructions=research_instructions,
    model=MODEL,
    tools=[WebSearchTool(search_context_size="low")],
    model_settings=ModelSettings(tool_choice="required")
)

architect_instructions = """
You are a master itinerary architect. You take research summaries and user preferences 
to create a beautiful, cohesive, and practical day-by-day travel plan.
"""
architect_agent = Agent(
    name="Architect Agent",
    instructions=architect_instructions,
    model=MODEL,
    output_type=TravelItinerary
)

##  Tools

In [ ]:
@function_tool
def send_itinerary_email(to_email: str, subject: str, itinerary_html: str) -> Dict[str, str]:
    """ Sends the final travel itinerary as a beautiful HTML email """
    try:
        print(f"Sending itinerary to {to_email}...")
        sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
        from_email = Email("victorbarny4@gmail.com")
        content = Content("text/html", itinerary_html)
        mail = Mail(from_email, To(to_email), subject, content).get()
        response = sg.client.mail.send.post(request_body=mail)
        return {"status": "success", "code": response.status_code}
    except Exception as e:
        return {"status": "error", "message": str(e)}

## Orchestration Logic

In [ ]:
async def perform_parallel_research(destination: str, interests: List[str]):
    """ Uses asyncio to run research tasks in parallel for efficiency """
    print(f"🔍 Starting research for {destination}...")
    
    async def research_topic(topic: str):
        query = f"Best {topic} in {destination} for travellers interested in {', '.join(interests)}"
        try:
            result = await Runner.run(research_agent, query)
            return result.final_output
        except Exception as e:
            return f"Error researching {topic}: {str(e)}"

    topics = ["attractions", "accommodations", "logistics & transport", "local food"]
    tasks = [research_topic(t) for t in topics]
    
    summaries = await asyncio.gather(*tasks)
    return "\n\n".join(summaries)

async def manage_travel_plan(user_input: str, history: List[Dict] = []):
    """ The main coordination flow """
    try: 
        print("💬 Consulting Inquiry Agent...")
        engagement_result = await Runner.run(inquiry_agent, user_input, history=history)
        engagement = engagement_result.final_output
        
        if not engagement.is_planning_complete:
            return f"Travel Planner: I still need some more details. {engagement_result.final_output}"
        
        # 2. Research Phase 
        research_summary = await perform_parallel_research(engagement.destination, engagement.interests)
        
        # 3. Architect Phase
        print("Designing Itinerary...")
        itinerary_result = await Runner.run(architect_agent, f"User Preferences: {engagement}\nResearch: {research_summary}")
        itinerary = itinerary_result.final_output
        
        # 4. Result Formatting (Simplified for demo)
        return f"Itinerary Created for {itinerary.destination}!\n\nHighlights: {', '.join(itinerary.highlights)}"
        
    except Exception as e:
        return f"Error in travel planning: {str(e)}"

## Execution Trace
To try this out, run the cell below. Remember to check your [OpenAI Trace Dashboard](https://platform.openai.com/traces)!

In [ ]:
async def main():
    # Example usage
    prompt = "I want to plan a 3-day luxury trip to Tokyo in May. I love food and history."
    with trace("Travel Planning Flow"):
        response = await manage_travel_plan(prompt)
        print("\n" + response)

if __name__ == "__main__":
    await main()